# Set1-5: a nested feature hierarchy shared across RSQ1, RSQ2, and RSQ3

**What this notebook is for.** Three different questions in this project each need their own
answer to "which environmental/management variables are worth including":

- **RSQ1** -- Avenue 1's predictive tiers (feeds `dnn_env_terrain`/`pinn_env_terrain*`, target
  `elev_percentile_95th`).
- **RSQ2** -- Avenue 1's attribution screen (feeds Elastic Net/XGBoost/NLME, target
  `mean_cr_residual`).
- **RSQ3** -- Avenue 2's scope system (feeds Elastic Net/XGBoost/GNNWR, target
  `local_y_max_difference`).

This notebook builds the SAME five-set recipe for all three, using
`models/xgb_environmental/feature_set_builder.py`:

- **Set1** = baseline only (the 5 stand/management columns every model already gets).
- **Set2** = Set1 + top 10 deduplicated candidates, by COMBINED importance rank (see below).
- **Set3** = Set1 + terrain/wind candidates in the top half of their own category by combined rank.
- **Set4** = Set1 + every remaining category's candidates in the top half of their own category.
- **Set5** = Set1 + every deduplicated candidate, gate not applied.

**Importance signal: three methods combined, not one.** A single ranking method has a blind spot
-- univariate correlation ignores everything else already in the model; permutation importance is
known to be biased under correlated predictors; drop-column ablation depends on one particular
model/split. `combine_importance_ranks()` rank-aggregates all three (Spearman `|rho|`, XGBoost
permutation importance, XGBoost drop-column ablation R2 change) -- a variable has to look
important across multiple different lenses to rank highly overall, not just fool one method. The
three are computed by `rank_by_target_correlation`, `permutation_importance_ranking`, and
`drop_column_ablation_ranking` respectively (all in `feature_set_builder.py`).

Permutation importance and drop-column ablation both need a held-out split to mean anything --
this notebook uses ONE `spatial_block_split` per RSQ (not the project's real pooled 5-fold CV,
which is a heavier, separate step reserved for evaluating whichever Set actually gets chosen,
not for screening candidates in the first place). Still spatially blocked, not a plain random
split, matching this project's standard discipline throughout.

**A real bug this notebook fixes.** The existing `stage1_terrain`/`stage2_terrain_wind`/
`stage3_terrain_wind_plus`/`stage4_all_environmental` tiers in `ENV_TERRAIN_FEATURE_SETS` (built
by `multicollinearity_screen_av1.ipynb`) feed RSQ1's models, but were gated using correlation
against `mean_cr_residual` -- RSQ2's target, not RSQ1's own `elev_percentile_95th`. Every ranking/
gate call in this notebook takes `target_column` as an explicit argument specifically so that
mismatch can't happen again by accident.

**Scope.** Built for the 4survey cohort (this project's primary cohort throughout, matching both
existing sibling notebooks). 6survey gets one informational re-run near the end (does Set2's
top-5 change?), not a second full deliverable.

**Whole-pipeline design decisions already agreed before this notebook was written** (not
re-litigated here):
1. Set1 (baseline) is unconditionally present in every set, all three RSQs -- including RSQ3,
   where the existing `SCOPE_GROUPS` deliberately has NO fixed baseline (so management's own
   marginal contribution stays visible via `terrain_wind` vs `terrain_wind_plus_management`).
   That existing scope system is untouched by this notebook, kept alongside these new sets, not
   replaced by them.
2. `thinning_status` is excluded from Set1 (matches existing precedent: both
   `CATEGORY_GROUPS["stand_structure"]` and `MANAGEMENT_COLUMNS` already exclude it).
3. `Thin` is excluded from Set1 -- confirmed directly against the real data that
   `Thin + time_since_thinning_missing == 1.0` EXACTLY for all 71,766 plots (a genuine
   deterministic duplicate, not a VIF nuisance). `time_since_thinning_missing` is kept: its actual
   job is as a missingness flag paired with `time_since_thinning`, so the two of them together
   already carry everything `Thin` does, losslessly.
4. `whcl` (windthrow hazard class) stays in the wind category, never baseline/management, despite
   coming from the same raw GPKG survey as the stand/management columns -- it's conceptually a
   wind-exposure measurement, not a management action.
5. ALL THREE RSQs get a VIF pass (threshold 5.0) on Set3/4/5 before those are treated as final
   -- not just RSQ2. Extended after two pieces of project-specific evidence, not just theory:
   `dnn_env_terrain`'s own earlier tier sweep showed held-out R2 degrading as tiers widened with
   more redundant columns (RSQ1's exported Set2 independently checked: 4-of-5 columns VIF>=5);
   RSQ3 feeds Elastic Net + GNNWR, both linear-coefficient models exactly as VIF-sensitive as
   RSQ2's Elastic Net/NLME (RSQ3's Set3 independently checked: 5-of-9 non-baseline columns
   VIF>=5, before even counting the categorical dummy-trap fixed in step 2 below).
6. RSQ3's categorical one-hot encoding (`prepare_broad_table`) keeps every level of
   `ceh_pedotope`/`ceh_subsurface_drainage`/`ceh_textural_composition` -- correct for XGBoost/
   GNNWR, but a design-matrix singularity for VIF (one categorical's dummies structurally sum to
   a constant). `drop_reference_level_per_category` removes one level per categorical from RSQ3's
   candidate pool BEFORE dedup/ranking/VIF ever run, so every signal sees one consistent
   representation -- confirmed necessary directly: RSQ3's Set5 showed `VIF=inf` for multiple
   `ceh_*` dummy columns before this fix. `prepare_broad_table` itself, and the existing
   `SCOPE_GROUPS` system that also calls it, are untouched.


In [1]:
import sys
from pathlib import Path

notebook_directory = Path.cwd().resolve()
project_root = next(
    folder for folder in [notebook_directory, *notebook_directory.parents]
    if (folder / "README.md").exists() and (folder / "data").exists()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

from models.xgb_environmental.data import load_plots_for_cohort
from models.xgb_environmental.xgb_environmental import FEATURE_PROVENANCE, ALL_FEATURE_COLUMNS
from models.xgb_environmental.grouped_analysis import CATEGORY_GROUPS
from models.xgb_environmental.feature_set_builder import (
    SET1_BASELINE_COLUMNS,
    dedup_candidates,
    drop_reference_level_per_category,
    rank_by_target_correlation,
    permutation_importance_ranking,
    drop_column_ablation_ranking,
    combine_importance_ranks,
    gate_columns_by_combined_rank,
    build_set2,
    build_set3,
    build_set4,
    build_set5,
    run_vif_pass,
    strip_baseline_for_export,
)
from models.common.torch_data import (
    ENV_TERRAIN_FEATURE_SETS,
    TARGET_COLUMN as RSQ1_TARGET,
    load_split_table_with_terrain,
    assert_env_terrain_features_disjoint_from_noenv,
)
from models.growth_curve_attribution.broad_environmental_check import (
    FEATURE_GROUPS,
    columns_for_groups,
    prepare_broad_table,
)
from models.growth_curve_attribution.scale_comparison_check import TARGET as RSQ3_TARGET

RSQ2_TARGET = "mean_cr_residual"
COHORT = "4survey"

pd.set_option("display.max_rows", 150)
pd.set_option("display.width", 160)

print(f"RSQ1 target: {RSQ1_TARGET}")
print(f"RSQ2 target: {RSQ2_TARGET}")
print(f"RSQ3 target: {RSQ3_TARGET}")
print(f"Set1 baseline (all three RSQs): {SET1_BASELINE_COLUMNS}")
print(f"Cohort: {COHORT}")


RSQ1 target: elev_percentile_95th
RSQ2 target: mean_cr_residual
RSQ3 target: local_y_max_difference
Set1 baseline (all three RSQs): ['CanopyCover', 'time_since_thinning', 'time_since_thinning_missing', 'recent_thinning_5yr']
Cohort: 4survey


## 1. Target-circularity check

Already confirmed independently by both existing sibling notebooks
(`multicollinearity_screen_av1.ipynb`, `multicollinearity_screen_av2_reference.ipynb`), restated
once here for all three targets:

- **RSQ1** (`elev_percentile_95th`): raw LiDAR height, untouched by any environmental variable.
- **RSQ2** (`mean_cr_residual`): `y_max_fit - y_max_yldc`, built only from height/Age survey data
  and the Forestry Commission's static yield-class table.
- **RSQ3** (`local_y_max_difference`): same target-construction code path as RSQ2's
  `build_plot_level_table()`, same conclusion.

No environmental/management candidate appears in the construction of any of the three -- pass,
not re-derived here.


## 2. Candidate pools and tables, per RSQ

RSQ1 and RSQ2 share one candidate pool: `ALL_FEATURE_COLUMNS` (the current, leak-fixed 37-variable
pool) minus Set1's baseline minus the three unordered categorical class-ID columns (never valid
as continuous Spearman/VIF input). RSQ3 uses Avenue 2's own `FEATURE_GROUPS`
(`terrain_wind`/`climate`/`soil_site`/`edge_position`, `management` excluded -- that's Set1's job
here, not a group to dedup/rank), with categoricals one-hot expanded by `prepare_broad_table`.

Every table here is the FULL population (no train/test split) for the dedup/Spearman steps --
those are screening steps, not performance evaluations. The permutation/ablation steps (section
4) each build their OWN spatial train/test split internally, since those specifically need one.


In [2]:
CATEGORICAL_COLUMNS = ["ceh_pedotope", "ceh_subsurface_drainage", "ceh_textural_composition"]

# Excludes the FULL 5-column stand_structure group (not just the 4-column SET1_BASELINE_COLUMNS)
# -- Thin belongs in neither: it's not part of the deduped baseline (see feature_set_builder.py's
# own reasoning), and it's also not a real "environmental candidate" to be ranked/gated, it's the
# same management concept as time_since_thinning_missing under a different name. Subtracting only
# SET1_BASELINE_COLUMNS here would let Thin leak back in as if it were an environmental variable
# -- confirmed by hitting exactly that: torch_data.py's own disjointness guard rejected it, since
# Thin is still part of that file's own, separate, unchanged production baseline
# (BINARY_PASSTHROUGH_COLUMNS).
shared_candidate_pool = [
    column for column in ALL_FEATURE_COLUMNS
    if column not in CATEGORY_GROUPS["stand_structure"] and column not in CATEGORICAL_COLUMNS
]
print(f"Shared RSQ1/RSQ2 candidate pool: {len(shared_candidate_pool)} columns")
print(shared_candidate_pool)


Shared RSQ1/RSQ2 candidate pool: 40 columns
['elevation', 'slope_degrees', 'northness', 'eastness', 'profile_curvature', 'plan_curvature', 'tpi', 'elevation_roughness', 'tpi_500m', 'local_relief_500m', 'solar_radiation_index', 'inverse_slope_proxy', 'frost_hollow_flag', 'topex', 'windward_topex', 'gwa_weibull_a_10m', 'gwa_weibull_k_10m', 'gwa_wind_p95_10m', 'gwa_prob_above_critical_10m', 'gwa_wind_speed_50m', 'gwa_weibull_a_50m', 'gwa_weibull_k_50m', 'gwa_wind_p95_50m', 'gwa_prob_above_critical_50m', 'dist_to_cpmt_boundary', 'dist_to_forest_perimeter', 'dist_to_scpt_boundary', 'dist_to_block_boundary', 'cpmt_compactness_ratio', 'dist_to_road', 'dist_to_watercourse', 'gwa_wind_speed_10m', 'soilgrids_ph', 'ceh_twi', 'chelsa_bio1_celsius', 'chelsa_gdd5_degc', 'chelsa_bio12_precip_mm', 'tas_mean', 'groundfrost_mean', 'whcl']


In [3]:
# RSQ1 table. load_split_table_with_terrain() already handles the documented whcl merge-collision
# fix and cohort-suffix resolution (tas_mean_4survey etc.), and its own disjointness assert passes
# automatically here since shared_candidate_pool already excludes every baseline column.
rsq1_table = load_split_table_with_terrain(COHORT, "spatial_block", shared_candidate_pool)
print(f"RSQ1 table: {len(rsq1_table):,} rows")


  Rows before filtering: 287,064
  Removed by Age < 30 at the 2023 survey (whole plot dropped): 54,616 rows


  Removed by yield class between 2 and 50 (whole plot dropped): 0 rows
  Rows after filtering: 232,448


  Buffer: 2,186 train plots are within 60 m of a val/test plot and are excluded ('buffer'). val/test plots are never removed.


RSQ1 table: 230,728 rows


In [4]:
# RSQ2 table. load_plots_for_cohort() already resolves mean_cr_residual/tas_mean/groundfrost_mean
# cohort-suffixed columns to their plain names.
rsq2_table = load_plots_for_cohort(COHORT)
print(f"RSQ2 table: {len(rsq2_table):,} plots")


RSQ2 table: 71,766 plots


In [5]:
# RSQ3 table. management group excluded from the SCREENING pool on purpose -- it's Set1's
# baseline here, not a group to dedup/rank/gate. But the table itself still needs the baseline
# columns present as real data (permutation/ablation importance build a "baseline + candidates"
# model, so it needs both) -- prepare_broad_table() only merges in whatever raw_feature_columns
# it's given, so SET1_BASELINE_COLUMNS has to be explicitly requested here even though it's
# excluded again immediately after, from rsq3_model_columns specifically (which stays
# screening-only, unchanged).
rsq3_raw_columns = columns_for_groups(["terrain_wind", "climate", "soil_site", "edge_position"])
rsq3_table, rsq3_all_model_columns = prepare_broad_table(COHORT, rsq3_raw_columns + SET1_BASELINE_COLUMNS)
rsq3_model_columns_with_baseline_excluded = [c for c in rsq3_all_model_columns if c not in SET1_BASELINE_COLUMNS]

# One reference level dropped per categorical (ceh_pedotope/ceh_subsurface_drainage/
# ceh_textural_composition) BEFORE dedup/ranking/VIF ever run -- see this notebook's own intro,
# point 6. Only affects this screening pool, not prepare_broad_table() itself.
CATEGORICAL_PREFIXES = ["ceh_pedotope=", "ceh_subsurface_drainage=", "ceh_textural_composition="]
rsq3_model_columns = drop_reference_level_per_category(rsq3_model_columns_with_baseline_excluded, CATEGORICAL_PREFIXES)
dropped_reference_levels = set(rsq3_model_columns_with_baseline_excluded) - set(rsq3_model_columns)

print(f"RSQ3 table: {len(rsq3_table):,} plots, {len(rsq3_model_columns)} model columns "
      f"(categoricals one-hot expanded, one reference level dropped per categorical, "
      f"baseline excluded from screening pool)")
print(f"Reference levels dropped: {sorted(dropped_reference_levels)}")


  Rows before filtering: 287,064
  Removed by Age < 30 at the 2023 survey (whole plot dropped): 54,616 rows
  Removed by yield class between 2 and 50 (whole plot dropped): 0 rows
  Rows after filtering: 232,448


  Rows before filtering: 287,064


  Removed by Age < 30 at the 2023 survey (whole plot dropped): 54,616 rows
  Removed by yield class between 2 and 50 (whole plot dropped): 0 rows
  Rows after filtering: 232,448


  Disturbance cleaning: excluded 315 of 56,841 plots (clearfell-like or measurement-inconsistent) before fitting y_max


  Broad complete-case population: dropped 412 of 56,526 plots
RSQ3 table: 56,114 plots, 41 model columns (categoricals one-hot expanded, one reference level dropped per categorical, baseline excluded from screening pool)
Reference levels dropped: ['ceh_pedotope=10.0', 'ceh_subsurface_drainage=1.0', 'ceh_textural_composition=1.0']


## 3. Dedup (stages 1-2): drop deterministic and near-exact (`|rho|>=0.95`) duplicates

Same two-stage logic as the existing `multicollinearity_screen_av1.ipynb`'s own `dedup_only()`
helper, now a reusable function (`feature_set_builder.dedup_candidates`) so all three RSQs call
identical code instead of three separate copies. No VIF yet -- that's a separate, later step.


In [6]:
rsq1_dedup, rsq1_dedup_log = dedup_candidates(rsq1_table, shared_candidate_pool, FEATURE_PROVENANCE, RSQ1_TARGET)
print(f"RSQ1 dedup: {len(shared_candidate_pool)} candidates -> {len(rsq1_dedup)} kept")
print(pd.DataFrame(rsq1_dedup_log).to_string(index=False) if rsq1_dedup_log else "(nothing dropped)")


RSQ1 dedup: 40 candidates -> 33 kept
                     column                     stage                                                                                reason
        inverse_slope_proxy 1_deterministic_duplicate            DERIVED from slope_degrees (= -slope_degrees exactly, not new information)
           gwa_wind_p95_10m 1_deterministic_duplicate         DERIVED from gwa_weibull_a_10m and gwa_weibull_k_10m: A * [-ln(1-0.95)]^(1/k)
gwa_prob_above_critical_10m 1_deterministic_duplicate                  DERIVED from gwa_weibull_a_10m and gwa_weibull_k_10m: exp[-(20/A)^k]
           gwa_wind_p95_50m 1_deterministic_duplicate DERIVED from gwa_weibull_a_50m and gwa_weibull_k_50m, same formula as the 10m version
gwa_prob_above_critical_50m 1_deterministic_duplicate DERIVED from gwa_weibull_a_50m and gwa_weibull_k_50m, same formula as the 10m version
         gwa_wind_speed_50m    2_near_exact_duplicate                                                       rho=1.000 with 

In [7]:
rsq2_dedup, rsq2_dedup_log = dedup_candidates(rsq2_table, shared_candidate_pool, FEATURE_PROVENANCE, RSQ2_TARGET)
print(f"RSQ2 dedup: {len(shared_candidate_pool)} candidates -> {len(rsq2_dedup)} kept")
print(pd.DataFrame(rsq2_dedup_log).to_string(index=False) if rsq2_dedup_log else "(nothing dropped)")


RSQ2 dedup: 40 candidates -> 33 kept
                     column                     stage                                                                                reason
        inverse_slope_proxy 1_deterministic_duplicate            DERIVED from slope_degrees (= -slope_degrees exactly, not new information)
           gwa_wind_p95_10m 1_deterministic_duplicate         DERIVED from gwa_weibull_a_10m and gwa_weibull_k_10m: A * [-ln(1-0.95)]^(1/k)
gwa_prob_above_critical_10m 1_deterministic_duplicate                  DERIVED from gwa_weibull_a_10m and gwa_weibull_k_10m: exp[-(20/A)^k]
           gwa_wind_p95_50m 1_deterministic_duplicate DERIVED from gwa_weibull_a_50m and gwa_weibull_k_50m, same formula as the 10m version
gwa_prob_above_critical_50m 1_deterministic_duplicate DERIVED from gwa_weibull_a_50m and gwa_weibull_k_50m, same formula as the 10m version
         gwa_wind_speed_50m    2_near_exact_duplicate                                                       rho=1.000 with 

In [8]:
rsq3_dedup, rsq3_dedup_log = dedup_candidates(rsq3_table, rsq3_model_columns, FEATURE_PROVENANCE, RSQ3_TARGET)
print(f"RSQ3 dedup: {len(rsq3_model_columns)} candidates -> {len(rsq3_dedup)} kept")
print(pd.DataFrame(rsq3_dedup_log).to_string(index=False) if rsq3_dedup_log else "(nothing dropped)")


RSQ3 dedup: 41 candidates -> 38 kept
                      column                  stage                           reason
         chelsa_bio1_celsius 2_near_exact_duplicate  rho=0.997 with its kept partner
ceh_textural_composition=4.0 2_near_exact_duplicate rho=-0.965 with its kept partner
           ceh_pedotope=12.0 2_near_exact_duplicate  rho=1.000 with its kept partner


## 4. Importance ranking: three signals, combined

For each RSQ: Spearman `|rho|` (cheap, whole population), XGBoost permutation importance, and
XGBoost drop-column ablation (both on one spatial train/test split) -- then rank-aggregated into
one combined table. Set2 = baseline + this combined ranking's top 10.


In [9]:
# control_columns=["Age"]: Age is NOT circular for RSQ1 (unlike RSQ2/RSQ3, whose targets are
# residuals already built FROM Age) -- it's this project's own single dominant predictor of raw
# height. Without it, the reference model is badly misspecified (first real run: R2=-0.168, worse
# than predicting the mean). Age is included in the reference model so importance is measured
# correctly (given the model already has Age, which the real dnn_env_terrain model does too, via
# its own separate pathway), but never appears in the ranking or the exported Set2-5 lists.
rsq1_spearman = rank_by_target_correlation(rsq1_table, rsq1_dedup, RSQ1_TARGET)
rsq1_permutation, rsq1_permutation_baseline_r2 = permutation_importance_ranking(rsq1_table, rsq1_dedup, SET1_BASELINE_COLUMNS, RSQ1_TARGET, control_columns=["Age"])
rsq1_ablation, rsq1_ablation_full_r2 = drop_column_ablation_ranking(rsq1_table, rsq1_dedup, SET1_BASELINE_COLUMNS, RSQ1_TARGET, control_columns=["Age"])
rsq1_combined = combine_importance_ranks(rsq1_spearman, rsq1_permutation, rsq1_ablation)

print(f"RSQ1 reference model (Age included as a control column): permutation baseline R2={rsq1_permutation_baseline_r2:.4f}, ablation full R2={rsq1_ablation_full_r2:.4f}")
print("\nRSQ1 combined importance ranking (all candidates):")
print(rsq1_combined.to_string(index=False))

rsq1_set2, rsq1_set2_skip_log = build_set2(rsq1_table, rsq1_combined, SET1_BASELINE_COLUMNS, top_n=10)
print(f"\nRSQ1 Set2 (VIF-screened, backfilled): {rsq1_set2}")
print(f"RSQ1 Set2 candidates skipped for VIF: {rsq1_set2_skip_log if rsq1_set2_skip_log else '(none)'}")


  Buffer: 610 train plots are within 60 m of a val/test plot and are excluded ('buffer'). val/test plots are never removed.


  Buffer: 610 train plots are within 60 m of a val/test plot and are excluded ('buffer'). val/test plots are never removed.


RSQ1 reference model (Age included as a control column): permutation baseline R2=0.2550, ablation full R2=0.2550

RSQ1 combined importance ranking (all candidates):
                variable  abs_rho  r2_drop_from_permutation  r2_drop_from_removal  spearman_rank  permutation_rank  ablation_rank  average_rank
       gwa_weibull_k_50m 0.202099                  0.007715              0.075749            7.0              13.0            5.0      8.333333
   dist_to_scpt_boundary 0.117468                  0.008253              0.114558           15.0              10.0            2.0      9.000000
               elevation 0.240352                  0.039047             -0.003975            3.0               1.0           27.0     10.333333
   dist_to_cpmt_boundary 0.096569                  0.016946              0.030031           19.0               3.0           11.0     11.000000
                tas_mean 0.199924                  0.008177              0.020647            8.0              11.0 

/Users/shreeyagorasia/UoE_Docs/Dissertation/forest_diss/.venv/lib/python3.13/site-packages/statsmodels/regression/linear_model.py:1782: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.centered_tss
/Users/shreeyagorasia/UoE_Docs/Dissertation/forest_diss/.venv/lib/python3.13/site-packages/statsmodels/regression/linear_model.py:1782: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.centered_tss
/Users/shreeyagorasia/UoE_Docs/Dissertation/forest_diss/.venv/lib/python3.13/site-packages/statsmodels/regression/linear_model.py:1782: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.centered_tss
/Users/shreeyagorasia/UoE_Docs/Dissertation/forest_diss/.venv/lib/python3.13/site-packages/statsmodels/regression/linear_model.py:1782: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.centered_tss
/Users/shreeyagorasia/UoE_Docs/Dissertation/forest_diss/.ven

/Users/shreeyagorasia/UoE_Docs/Dissertation/forest_diss/.venv/lib/python3.13/site-packages/statsmodels/regression/linear_model.py:1782: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.centered_tss
/Users/shreeyagorasia/UoE_Docs/Dissertation/forest_diss/.venv/lib/python3.13/site-packages/statsmodels/regression/linear_model.py:1782: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.centered_tss
/Users/shreeyagorasia/UoE_Docs/Dissertation/forest_diss/.venv/lib/python3.13/site-packages/statsmodels/regression/linear_model.py:1782: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.centered_tss


/Users/shreeyagorasia/UoE_Docs/Dissertation/forest_diss/.venv/lib/python3.13/site-packages/statsmodels/regression/linear_model.py:1782: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.centered_tss
/Users/shreeyagorasia/UoE_Docs/Dissertation/forest_diss/.venv/lib/python3.13/site-packages/statsmodels/regression/linear_model.py:1782: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.centered_tss


/Users/shreeyagorasia/UoE_Docs/Dissertation/forest_diss/.venv/lib/python3.13/site-packages/statsmodels/regression/linear_model.py:1782: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.centered_tss
/Users/shreeyagorasia/UoE_Docs/Dissertation/forest_diss/.venv/lib/python3.13/site-packages/statsmodels/regression/linear_model.py:1782: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.centered_tss



RSQ1 Set2 (VIF-screened, backfilled): ['CanopyCover', 'time_since_thinning', 'time_since_thinning_missing', 'recent_thinning_5yr', 'gwa_weibull_k_50m', 'dist_to_scpt_boundary', 'elevation', 'tas_mean', 'eastness', 'windward_topex', 'gwa_weibull_a_10m', 'gwa_weibull_a_50m', 'slope_degrees', 'cpmt_compactness_ratio']
RSQ1 Set2 candidates skipped for VIF: [{'column': 'dist_to_cpmt_boundary', 'vif_if_added': 5.97, 'reason': 'VIF=5.97 > 5.0 against baseline + already-kept Set2 candidates'}, {'column': 'dist_to_block_boundary', 'vif_if_added': 5.23, 'reason': 'VIF=5.23 > 5.0 against baseline + already-kept Set2 candidates'}, {'column': 'chelsa_gdd5_degc', 'vif_if_added': 5.11, 'reason': 'VIF=5.11 > 5.0 against baseline + already-kept Set2 candidates'}]


/Users/shreeyagorasia/UoE_Docs/Dissertation/forest_diss/.venv/lib/python3.13/site-packages/statsmodels/regression/linear_model.py:1782: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.centered_tss


In [10]:
rsq2_spearman = rank_by_target_correlation(rsq2_table, rsq2_dedup, RSQ2_TARGET)
rsq2_permutation, rsq2_permutation_baseline_r2 = permutation_importance_ranking(rsq2_table, rsq2_dedup, SET1_BASELINE_COLUMNS, RSQ2_TARGET)
rsq2_ablation, rsq2_ablation_full_r2 = drop_column_ablation_ranking(rsq2_table, rsq2_dedup, SET1_BASELINE_COLUMNS, RSQ2_TARGET)
rsq2_combined = combine_importance_ranks(rsq2_spearman, rsq2_permutation, rsq2_ablation)

print(f"RSQ2 reference model: permutation baseline R2={rsq2_permutation_baseline_r2:.4f}, ablation full R2={rsq2_ablation_full_r2:.4f}")
print("\nRSQ2 combined importance ranking (all candidates):")
print(rsq2_combined.to_string(index=False))

rsq2_set2, rsq2_set2_skip_log = build_set2(rsq2_table, rsq2_combined, SET1_BASELINE_COLUMNS, top_n=10)
print(f"\nRSQ2 Set2 (VIF-screened, backfilled): {rsq2_set2}")
print(f"RSQ2 Set2 candidates skipped for VIF: {rsq2_set2_skip_log if rsq2_set2_skip_log else '(none)'}")


  Buffer: 3,276 train plots are within 60 m of a val/test plot and are excluded ('buffer'). val/test plots are never removed.


  Buffer: 3,276 train plots are within 60 m of a val/test plot and are excluded ('buffer'). val/test plots are never removed.


RSQ2 reference model: permutation baseline R2=0.3691, ablation full R2=0.3691

RSQ2 combined importance ranking (all candidates):
                variable  abs_rho  r2_drop_from_permutation  r2_drop_from_removal  spearman_rank  permutation_rank  ablation_rank  average_rank
  chelsa_bio12_precip_mm 0.363904                  0.106964              0.060221            2.0               2.0           12.0      5.333333
        chelsa_gdd5_degc 0.310277                  0.032647              0.078154            5.0               5.0            8.0      6.000000
       gwa_weibull_k_50m 0.263665                  0.020188              0.152693            8.0               8.0            3.0      6.333333
           slope_degrees 0.086517                  0.045908              0.114971           17.0               3.0            6.0      8.666667
       gwa_weibull_a_10m 0.203203                  0.013670              0.117246           11.0              12.0            5.0      9.333333
      


RSQ2 Set2 (VIF-screened, backfilled): ['CanopyCover', 'time_since_thinning', 'time_since_thinning_missing', 'recent_thinning_5yr', 'chelsa_bio12_precip_mm', 'chelsa_gdd5_degc', 'gwa_weibull_k_50m', 'slope_degrees', 'gwa_weibull_a_10m', 'local_relief_500m', 'eastness', 'dist_to_road', 'gwa_weibull_a_50m', 'tas_mean']
RSQ2 Set2 candidates skipped for VIF: [{'column': 'elevation', 'vif_if_added': 9.65, 'reason': 'VIF=9.65 > 5.0 against baseline + already-kept Set2 candidates'}]


In [11]:
rsq3_spearman = rank_by_target_correlation(rsq3_table, rsq3_dedup, RSQ3_TARGET)
rsq3_permutation, rsq3_permutation_baseline_r2 = permutation_importance_ranking(rsq3_table, rsq3_dedup, SET1_BASELINE_COLUMNS, RSQ3_TARGET)
rsq3_ablation, rsq3_ablation_full_r2 = drop_column_ablation_ranking(rsq3_table, rsq3_dedup, SET1_BASELINE_COLUMNS, RSQ3_TARGET)
rsq3_combined = combine_importance_ranks(rsq3_spearman, rsq3_permutation, rsq3_ablation)

print(f"RSQ3 reference model: permutation baseline R2={rsq3_permutation_baseline_r2:.4f}, ablation full R2={rsq3_ablation_full_r2:.4f}")
print("\nRSQ3 combined importance ranking (all candidates):")
print(rsq3_combined.to_string(index=False))

rsq3_set2, rsq3_set2_skip_log = build_set2(rsq3_table, rsq3_combined, SET1_BASELINE_COLUMNS, top_n=10)
print(f"\nRSQ3 Set2 (VIF-screened, backfilled): {rsq3_set2}")
print(f"RSQ3 Set2 candidates skipped for VIF: {rsq3_set2_skip_log if rsq3_set2_skip_log else '(none)'}")


  Buffer: 2,158 train plots are within 60 m of a val/test plot and are excluded ('buffer'). val/test plots are never removed.


  Buffer: 2,158 train plots are within 60 m of a val/test plot and are excluded ('buffer'). val/test plots are never removed.


RSQ3 reference model: permutation baseline R2=0.0807, ablation full R2=0.0807

RSQ3 combined importance ranking (all candidates):
                    variable  abs_rho  r2_drop_from_permutation  r2_drop_from_removal  spearman_rank  permutation_rank  ablation_rank  average_rank
              windward_topex 0.155635                  0.046790             -0.028166            6.0               5.0           17.0      9.333333
                   elevation 0.169903                  0.060934             -0.046780            3.0               3.0           26.0     10.666667
            chelsa_gdd5_degc 0.128667                  0.022357             -0.029084            8.0               8.0           18.0     11.333333
      cpmt_compactness_ratio 0.054873                  0.019129              0.001533           18.0              11.0            7.0     12.000000
          gwa_wind_speed_50m 0.177987                  0.013602             -0.038458            1.0              16.0           2


RSQ3 Set2 (VIF-screened, backfilled): ['CanopyCover', 'time_since_thinning', 'time_since_thinning_missing', 'recent_thinning_5yr', 'windward_topex', 'elevation', 'cpmt_compactness_ratio', 'gwa_wind_speed_50m', 'slope_degrees', 'tpi_500m', 'dist_to_road', 'topex', 'solar_radiation_index', 'soilgrids_ph']
RSQ3 Set2 candidates skipped for VIF: [{'column': 'chelsa_gdd5_degc', 'vif_if_added': 9.46, 'reason': 'VIF=9.46 > 5.0 against baseline + already-kept Set2 candidates'}, {'column': 'chelsa_bio12_precip_mm', 'vif_if_added': 5.4, 'reason': 'VIF=5.40 > 5.0 against baseline + already-kept Set2 candidates'}, {'column': 'tas_mean', 'vif_if_added': 6.04, 'reason': 'VIF=6.04 > 5.0 against baseline + already-kept Set2 candidates'}]


## 5. Set3/Set4: per-category gate by combined rank (top half of each category)

Terrain/wind candidates gated first, then everything else -- replacing the old fixed `|rho|>=0.10`
cutoff with "top half of this category's own candidates by combined rank," so the cutoff is
relative to the category's own members rather than an arbitrary absolute threshold.


In [12]:
terrain_wind_all = CATEGORY_GROUPS["terrain"] + CATEGORY_GROUPS["wind"]

rsq1_terrain_wind_dedup = [c for c in rsq1_dedup if c in terrain_wind_all]
rsq1_other_dedup = [c for c in rsq1_dedup if c not in terrain_wind_all]

rsq1_terrain_wind_gated, rsq1_terrain_wind_table = gate_columns_by_combined_rank(rsq1_combined, rsq1_terrain_wind_dedup)
rsq1_other_gated, rsq1_other_table = gate_columns_by_combined_rank(rsq1_combined, rsq1_other_dedup)

print("RSQ1 terrain/wind category, combined rank:")
print(rsq1_terrain_wind_table.to_string(index=False))
print(f"Passed (top half): {rsq1_terrain_wind_gated}")
print("\nRSQ1 other-category, combined rank:")
print(rsq1_other_table.to_string(index=False))
print(f"Passed (top half): {rsq1_other_gated}")

rsq1_set3 = build_set3(rsq1_terrain_wind_gated, SET1_BASELINE_COLUMNS)
rsq1_set4 = build_set4(rsq1_terrain_wind_gated, rsq1_other_gated, SET1_BASELINE_COLUMNS)
print(f"\nRSQ1 Set3: {rsq1_set3}")
print(f"RSQ1 Set4: {rsq1_set4}")


RSQ1 terrain/wind category, combined rank:
             variable  abs_rho  r2_drop_from_permutation  r2_drop_from_removal  spearman_rank  permutation_rank  ablation_rank  average_rank
    gwa_weibull_k_50m 0.202099                  0.007715              0.075749            7.0              13.0            5.0      8.333333
            elevation 0.240352                  0.039047             -0.003975            3.0               1.0           27.0     10.333333
             eastness 0.051543                  0.016403              0.081636           27.0               4.0            4.0     11.666667
       windward_topex 0.164849                  0.031430             -0.001003            9.0               2.0           25.0     12.000000
    gwa_weibull_a_10m 0.119599                  0.012157              0.013067           13.0               6.0           20.0     13.000000
    gwa_weibull_a_50m 0.243688                 -0.003011              0.023425            2.0              26.0

In [13]:
rsq2_terrain_wind_dedup = [c for c in rsq2_dedup if c in terrain_wind_all]
rsq2_other_dedup = [c for c in rsq2_dedup if c not in terrain_wind_all]

rsq2_terrain_wind_gated, rsq2_terrain_wind_table = gate_columns_by_combined_rank(rsq2_combined, rsq2_terrain_wind_dedup)
rsq2_other_gated, rsq2_other_table = gate_columns_by_combined_rank(rsq2_combined, rsq2_other_dedup)

print("RSQ2 terrain/wind category, combined rank:")
print(rsq2_terrain_wind_table.to_string(index=False))
print(f"Passed (top half): {rsq2_terrain_wind_gated}")
print("\nRSQ2 other-category, combined rank:")
print(rsq2_other_table.to_string(index=False))
print(f"Passed (top half): {rsq2_other_gated}")

rsq2_set3 = build_set3(rsq2_terrain_wind_gated, SET1_BASELINE_COLUMNS)
rsq2_set4 = build_set4(rsq2_terrain_wind_gated, rsq2_other_gated, SET1_BASELINE_COLUMNS)
print(f"\nRSQ2 Set3 (pre-VIF): {rsq2_set3}")
print(f"RSQ2 Set4 (pre-VIF): {rsq2_set4}")


RSQ2 terrain/wind category, combined rank:
             variable  abs_rho  r2_drop_from_permutation  r2_drop_from_removal  spearman_rank  permutation_rank  ablation_rank  average_rank
    gwa_weibull_k_50m 0.263665                  0.020188              0.152693            8.0               8.0            3.0      6.333333
        slope_degrees 0.086517                  0.045908              0.114971           17.0               3.0            6.0      8.666667
    gwa_weibull_a_10m 0.203203                  0.013670              0.117246           11.0              12.0            5.0      9.333333
            elevation 0.352501                  0.151193              0.001629            3.0               1.0           26.0     10.000000
    local_relief_500m 0.153843                  0.026888              0.050799           13.0               7.0           15.0     11.666667
             eastness 0.071021                  0.010323              0.132158           19.0              13.0

In [14]:
rsq3_terrain_wind_group = FEATURE_GROUPS["terrain_wind"]
rsq3_terrain_wind_dedup = [c for c in rsq3_dedup if c in rsq3_terrain_wind_group]
rsq3_other_dedup = [c for c in rsq3_dedup if c not in rsq3_terrain_wind_group]

rsq3_terrain_wind_gated, rsq3_terrain_wind_table = gate_columns_by_combined_rank(rsq3_combined, rsq3_terrain_wind_dedup)
rsq3_other_gated, rsq3_other_table = gate_columns_by_combined_rank(rsq3_combined, rsq3_other_dedup)

print("RSQ3 terrain/wind category, combined rank:")
print(rsq3_terrain_wind_table.to_string(index=False))
print(f"Passed (top half): {rsq3_terrain_wind_gated}")
print("\nRSQ3 other-category, combined rank:")
print(rsq3_other_table.to_string(index=False))
print(f"Passed (top half): {rsq3_other_gated}")

rsq3_set3 = build_set3(rsq3_terrain_wind_gated, SET1_BASELINE_COLUMNS)
rsq3_set4 = build_set4(rsq3_terrain_wind_gated, rsq3_other_gated, SET1_BASELINE_COLUMNS)
print(f"\nRSQ3 Set3: {rsq3_set3}")
print(f"RSQ3 Set4: {rsq3_set4}")


RSQ3 terrain/wind category, combined rank:
             variable  abs_rho  r2_drop_from_permutation  r2_drop_from_removal  spearman_rank  permutation_rank  ablation_rank  average_rank
       windward_topex 0.155635                  0.046790             -0.028166            6.0               5.0           17.0      9.333333
            elevation 0.169903                  0.060934             -0.046780            3.0               3.0           26.0     10.666667
   gwa_wind_speed_50m 0.177987                  0.013602             -0.038458            1.0              16.0           20.0     12.333333
        slope_degrees 0.101046                  0.064743             -0.042366           12.0               2.0           24.0     12.666667
             tpi_500m 0.096044                  0.048902             -0.039372           13.0               4.0           21.0     12.666667
                topex 0.120220                  0.020876             -0.043712           10.0              10.0

## 6. Set5: baseline + every deduplicated candidate, gate not applied


In [15]:
rsq1_set5 = build_set5(rsq1_dedup, SET1_BASELINE_COLUMNS)
rsq2_set5 = build_set5(rsq2_dedup, SET1_BASELINE_COLUMNS)
rsq3_set5 = build_set5(rsq3_dedup, SET1_BASELINE_COLUMNS)
print(f"RSQ1 Set5: {rsq1_set5}")
print(f"RSQ2 Set5 (pre-VIF): {rsq2_set5}")
print(f"RSQ3 Set5: {rsq3_set5}")


RSQ1 Set5: ['CanopyCover', 'time_since_thinning', 'time_since_thinning_missing', 'recent_thinning_5yr', 'elevation', 'slope_degrees', 'northness', 'eastness', 'profile_curvature', 'plan_curvature', 'tpi', 'elevation_roughness', 'tpi_500m', 'local_relief_500m', 'solar_radiation_index', 'frost_hollow_flag', 'topex', 'windward_topex', 'gwa_weibull_a_10m', 'gwa_weibull_k_10m', 'gwa_weibull_a_50m', 'gwa_weibull_k_50m', 'dist_to_cpmt_boundary', 'dist_to_forest_perimeter', 'dist_to_scpt_boundary', 'dist_to_block_boundary', 'cpmt_compactness_ratio', 'dist_to_road', 'dist_to_watercourse', 'gwa_wind_speed_10m', 'soilgrids_ph', 'ceh_twi', 'chelsa_gdd5_degc', 'chelsa_bio12_precip_mm', 'tas_mean', 'groundfrost_mean', 'whcl']
RSQ2 Set5 (pre-VIF): ['CanopyCover', 'time_since_thinning', 'time_since_thinning_missing', 'recent_thinning_5yr', 'elevation', 'slope_degrees', 'northness', 'eastness', 'profile_curvature', 'plan_curvature', 'tpi', 'elevation_roughness', 'tpi_500m', 'local_relief_500m', 'solar_

## 7. VIF pass (all three RSQs) on Set3/Set4/Set5

Pairwise dedup (section 3) alone doesn't catch 3-or-more-variable collinearity, and rank-
aggregated importance (section 4) says nothing about redundancy between variables that both
independently look important. Extended to all three RSQs (see this notebook's own intro, point
5) -- run_vif_pass() protects baseline from removal in every case (baseline stays in the design
matrix, so it still affects every other column's VIF number, it's just never the one dropped).


In [16]:
def print_vif_pass(rsq_name, table, set_name, columns):
    final_columns, vif_log = run_vif_pass(table, columns)
    print(f"{rsq_name} {set_name}: {len(columns)} -> {len(final_columns)} after VIF")
    print(pd.DataFrame(vif_log).to_string(index=False) if vif_log else "(nothing dropped)")
    return final_columns


rsq1_set3_final = print_vif_pass("RSQ1", rsq1_table, "Set3", rsq1_set3)
rsq1_set4_final = print_vif_pass("RSQ1", rsq1_table, "Set4", rsq1_set4)
rsq1_set5_final = print_vif_pass("RSQ1", rsq1_table, "Set5", rsq1_set5)
print(f"\nRSQ1 Set3 (final): {rsq1_set3_final}")
print(f"RSQ1 Set4 (final): {rsq1_set4_final}")
print(f"RSQ1 Set5 (final): {rsq1_set5_final}")


/Users/shreeyagorasia/UoE_Docs/Dissertation/forest_diss/.venv/lib/python3.13/site-packages/statsmodels/regression/linear_model.py:1782: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.centered_tss


RSQ1 Set3: 15 -> 15 after VIF
(nothing dropped)


/Users/shreeyagorasia/UoE_Docs/Dissertation/forest_diss/.venv/lib/python3.13/site-packages/statsmodels/regression/linear_model.py:1782: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.centered_tss


/Users/shreeyagorasia/UoE_Docs/Dissertation/forest_diss/.venv/lib/python3.13/site-packages/statsmodels/regression/linear_model.py:1782: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.centered_tss


/Users/shreeyagorasia/UoE_Docs/Dissertation/forest_diss/.venv/lib/python3.13/site-packages/statsmodels/regression/linear_model.py:1782: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.centered_tss


...

/Users/shreeyagorasia/UoE_Docs/Dissertation/forest_diss/.venv/lib/python3.13/site-packages/statsmodels/regression/linear_model.py:1782: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.centered_tss


/Users/shreeyagorasia/UoE_Docs/Dissertation/forest_diss/.venv/lib/python3.13/site-packages/statsmodels/regression/linear_model.py:1782: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.centered_tss


/Users/shreeyagorasia/UoE_Docs/Dissertation/forest_diss/.venv/lib/python3.13/site-packages/statsmodels/regression/linear_model.py:1782: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.centered_tss


/Users/shreeyagorasia/UoE_Docs/Dissertation/forest_diss/.venv/lib/python3.13/site-packages/statsmodels/regression/linear_model.py:1782: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.centered_tss


RSQ1 Set5: 37 -> 30 after VIF
                column stage                                                              reason
             elevation 3_vif VIF=16.77 against the other 36 remaining candidates (threshold 5.0)
 dist_to_cpmt_boundary 3_vif VIF=15.72 against the other 35 remaining candidates (threshold 5.0)
                   tpi 3_vif  VIF=9.88 against the other 34 remaining candidates (threshold 5.0)
   elevation_roughness 3_vif  VIF=6.93 against the other 33 remaining candidates (threshold 5.0)
 solar_radiation_index 3_vif  VIF=5.98 against the other 32 remaining candidates (threshold 5.0)
              tpi_500m 3_vif  VIF=5.67 against the other 31 remaining candidates (threshold 5.0)
dist_to_block_boundary 3_vif  VIF=5.39 against the other 30 remaining candidates (threshold 5.0)

RSQ1 Set3 (final): ['CanopyCover', 'time_since_thinning', 'time_since_thinning_missing', 'recent_thinning_5yr', 'gwa_weibull_k_50m', 'elevation', 'eastness', 'windward_topex', 'gwa_weibull_a_10

In [17]:
rsq2_set3_final = print_vif_pass("RSQ2", rsq2_table, "Set3", rsq2_set3)
rsq2_set4_final = print_vif_pass("RSQ2", rsq2_table, "Set4", rsq2_set4)
rsq2_set5_final = print_vif_pass("RSQ2", rsq2_table, "Set5", rsq2_set5)
print(f"\nRSQ2 Set3 (final): {rsq2_set3_final}")
print(f"RSQ2 Set4 (final): {rsq2_set4_final}")
print(f"RSQ2 Set5 (final): {rsq2_set5_final}")

print(f"\nRSQ2 table full row count: {len(rsq2_table):,}")
print(f"RSQ2 Set5 (widest) complete-case row count: {len(rsq2_table[rsq2_set5].dropna()):,}")


RSQ2 Set3: 15 -> 14 after VIF
   column stage                                                             reason
elevation 3_vif VIF=5.57 against the other 14 remaining candidates (threshold 5.0)


RSQ2 Set4: 21 -> 19 after VIF
          column stage                                                              reason
       elevation 3_vif VIF=31.83 against the other 20 remaining candidates (threshold 5.0)
chelsa_gdd5_degc 3_vif  VIF=7.84 against the other 19 remaining candidates (threshold 5.0)


RSQ2 Set5: 37 -> 28 after VIF
                column stage                                                              reason
             elevation 3_vif VIF=35.85 against the other 36 remaining candidates (threshold 5.0)
 dist_to_cpmt_boundary 3_vif VIF=19.74 against the other 35 remaining candidates (threshold 5.0)
   elevation_roughness 3_vif VIF=12.08 against the other 34 remaining candidates (threshold 5.0)
      chelsa_gdd5_degc 3_vif VIF=10.21 against the other 33 remaining candidates (threshold 5.0)
 dist_to_scpt_boundary 3_vif  VIF=7.14 against the other 32 remaining candidates (threshold 5.0)
                   tpi 3_vif  VIF=6.94 against the other 31 remaining candidates (threshold 5.0)
                 topex 3_vif  VIF=6.02 against the other 30 remaining candidates (threshold 5.0)
chelsa_bio12_precip_mm 3_vif  VIF=5.26 against the other 29 remaining candidates (threshold 5.0)
 solar_radiation_index 3_vif  VIF=5.06 against the other 28 remaining candidates (threshold 5.0)


In [18]:
rsq3_set3_final = print_vif_pass("RSQ3", rsq3_table, "Set3", rsq3_set3)
rsq3_set4_final = print_vif_pass("RSQ3", rsq3_table, "Set4", rsq3_set4)
rsq3_set5_final = print_vif_pass("RSQ3", rsq3_table, "Set5", rsq3_set5)
print(f"\nRSQ3 Set3 (final): {rsq3_set3_final}")
print(f"RSQ3 Set4 (final): {rsq3_set4_final}")
print(f"RSQ3 Set5 (final): {rsq3_set5_final}")


RSQ3 Set3: 13 -> 12 after VIF
             column stage                                                              reason
elevation_roughness 3_vif VIF=10.05 against the other 12 remaining candidates (threshold 5.0)


RSQ3 Set4: 24 -> 20 after VIF
               column stage                                                              reason
            elevation 3_vif VIF=36.36 against the other 23 remaining candidates (threshold 5.0)
  elevation_roughness 3_vif VIF=10.30 against the other 22 remaining candidates (threshold 5.0)
     chelsa_gdd5_degc 3_vif VIF=10.09 against the other 21 remaining candidates (threshold 5.0)
solar_radiation_index 3_vif  VIF=5.10 against the other 20 remaining candidates (threshold 5.0)


RSQ3 Set5: 42 -> 31 after VIF
                      column stage                                                              reason
                   elevation 3_vif VIF=39.85 against the other 41 remaining candidates (threshold 5.0)
       dist_to_cpmt_boundary 3_vif VIF=20.67 against the other 40 remaining candidates (threshold 5.0)
         elevation_roughness 3_vif VIF=13.51 against the other 39 remaining candidates (threshold 5.0)
            chelsa_gdd5_degc 3_vif VIF=11.82 against the other 38 remaining candidates (threshold 5.0)
ceh_textural_composition=2.0 3_vif VIF=10.32 against the other 37 remaining candidates (threshold 5.0)
      dist_to_block_boundary 3_vif  VIF=7.50 against the other 36 remaining candidates (threshold 5.0)
                         tpi 3_vif  VIF=6.75 against the other 35 remaining candidates (threshold 5.0)
                       topex 3_vif  VIF=6.28 against the other 34 remaining candidates (threshold 5.0)
      chelsa_bio12_precip_mm 3_vif  VIF=5.8

## 8. Summary: set sizes across all three RSQs


In [19]:
summary_rows = []
for rsq, sets in [
    ("RSQ1", {"Set1": SET1_BASELINE_COLUMNS, "Set2": rsq1_set2, "Set3": rsq1_set3_final, "Set4": rsq1_set4_final, "Set5": rsq1_set5_final}),
    ("RSQ2", {"Set1": SET1_BASELINE_COLUMNS, "Set2": rsq2_set2, "Set3": rsq2_set3_final, "Set4": rsq2_set4_final, "Set5": rsq2_set5_final}),
    ("RSQ3", {"Set1": SET1_BASELINE_COLUMNS, "Set2": rsq3_set2, "Set3": rsq3_set3_final, "Set4": rsq3_set4_final, "Set5": rsq3_set5_final}),
]:
    for set_name, columns in sets.items():
        summary_rows.append({"rsq": rsq, "set": set_name, "n_columns": len(columns)})
summary_df = pd.DataFrame(summary_rows)
print(summary_df.pivot(index="set", columns="rsq", values="n_columns"))


rsq   RSQ1  RSQ2  RSQ3
set                   
Set1     4     4     4
Set2    14    14    14
Set3    15    14    12
Set4    18    19    20
Set5    30    28    31


## 9. Sanity checks

(a) RSQ1's export-ready lists (baseline stripped) must pass the existing disjointness guard.
(b) How much did fixing the target-mismatch bug AND switching from Spearman-only to combined
importance ranking actually change, vs. the existing (wrongly targeted, Spearman-only)
`stage3_terrain_wind_plus`? (c) Hand-recompute a couple of RSQ2's Spearman values independently.


In [20]:
rsq1_export = {}
for set_name, columns in [
    ("nested_set2_top10", rsq1_set2), ("nested_set3_gated_terrain_wind_vif", rsq1_set3_final),
    ("nested_set4_gated_all_vif", rsq1_set4_final), ("nested_set5_all_ungated_vif", rsq1_set5_final),
]:
    export_columns = strip_baseline_for_export(columns, SET1_BASELINE_COLUMNS)
    assert_env_terrain_features_disjoint_from_noenv(export_columns)
    rsq1_export[set_name] = export_columns
print("RSQ1 export-ready lists (baseline stripped) all pass the disjointness check.")
for name, columns in rsq1_export.items():
    print(f"  {name} ({len(columns)}): {columns}")


RSQ1 export-ready lists (baseline stripped) all pass the disjointness check.
  nested_set2_top10 (10): ['gwa_weibull_k_50m', 'dist_to_scpt_boundary', 'elevation', 'tas_mean', 'eastness', 'windward_topex', 'gwa_weibull_a_10m', 'gwa_weibull_a_50m', 'slope_degrees', 'cpmt_compactness_ratio']
  nested_set3_gated_terrain_wind_vif (11): ['gwa_weibull_k_50m', 'elevation', 'eastness', 'windward_topex', 'gwa_weibull_a_10m', 'gwa_weibull_a_50m', 'slope_degrees', 'gwa_weibull_k_10m', 'solar_radiation_index', 'ceh_twi', 'gwa_wind_speed_10m']
  nested_set4_gated_all_vif (14): ['gwa_weibull_k_50m', 'eastness', 'windward_topex', 'gwa_weibull_a_10m', 'gwa_weibull_a_50m', 'slope_degrees', 'gwa_weibull_k_10m', 'solar_radiation_index', 'ceh_twi', 'gwa_wind_speed_10m', 'dist_to_scpt_boundary', 'tas_mean', 'chelsa_gdd5_degc', 'cpmt_compactness_ratio']
  nested_set5_all_ungated_vif (26): ['slope_degrees', 'northness', 'eastness', 'profile_curvature', 'plan_curvature', 'local_relief_500m', 'frost_hollow_flag

In [21]:
old_stage3 = set(ENV_TERRAIN_FEATURE_SETS["stage3_terrain_wind_plus"])
new_set3 = set(rsq1_export["nested_set3_gated_terrain_wind_vif"])
print(f"Old stage3_terrain_wind_plus, wrong target + per-category Spearman gate ({len(old_stage3)} columns):")
print(f"  {sorted(old_stage3)}")
print(f"New Set3, correct target + per-column combined-rank gate ({len(new_set3)} columns):")
print(f"  {sorted(new_set3)}")
print(f"Gained (in new, not old): {sorted(new_set3 - old_stage3)}")
print(f"Lost (in old, not new): {sorted(old_stage3 - new_set3)}")


Old stage3_terrain_wind_plus, wrong target + per-category Spearman gate (33 columns):
  ['ceh_twi', 'chelsa_bio12_precip_mm', 'chelsa_gdd5_degc', 'cpmt_compactness_ratio', 'dist_to_block_boundary', 'dist_to_cpmt_boundary', 'dist_to_forest_perimeter', 'dist_to_road', 'dist_to_scpt_boundary', 'dist_to_watercourse', 'eastness', 'elevation', 'elevation_roughness', 'frost_hollow_flag', 'groundfrost_mean', 'gwa_weibull_a_10m', 'gwa_weibull_a_50m', 'gwa_weibull_k_10m', 'gwa_weibull_k_50m', 'gwa_wind_speed_10m', 'local_relief_500m', 'northness', 'plan_curvature', 'profile_curvature', 'slope_degrees', 'soilgrids_ph', 'solar_radiation_index', 'tas_mean', 'topex', 'tpi', 'tpi_500m', 'whcl', 'windward_topex']
New Set3, correct target + per-column combined-rank gate (11 columns):
  ['ceh_twi', 'eastness', 'elevation', 'gwa_weibull_a_10m', 'gwa_weibull_a_50m', 'gwa_weibull_k_10m', 'gwa_weibull_k_50m', 'gwa_wind_speed_10m', 'slope_degrees', 'solar_radiation_index', 'windward_topex']
Gained (in new, n

In [22]:
for column in rsq2_spearman["variable"].head(3):
    values = rsq2_table[[column, RSQ2_TARGET]].dropna()
    manual_rho, _ = spearmanr(values[column], values[RSQ2_TARGET])
    printed_rho = rsq2_spearman.loc[rsq2_spearman["variable"] == column, "spearman_rho"].iloc[0]
    match = abs(printed_rho - manual_rho) < 1e-9
    print(f"{column}: printed={printed_rho:.4f}, hand-recomputed={manual_rho:.4f}, match={match}")


dist_to_road: printed=-0.3769, hand-recomputed=-0.3769, match=True
chelsa_bio12_precip_mm: printed=-0.3639, hand-recomputed=-0.3639, match=True
elevation: printed=-0.3525, hand-recomputed=-0.3525, match=True


## 10. Informational only: does Set2's top-5 change on the 6survey cohort?

Not a second full deliverable -- 4survey stays the primary, exported result (per the agreed
scope: 6survey is smaller, 7,467 rows, and noisier for correlation/VIF estimates). Spearman-only
here (not the full three-signal combination), as a cheap directional check.


In [23]:
six_rsq2_table = load_plots_for_cohort("6survey")
six_rsq2_dedup, _ = dedup_candidates(six_rsq2_table, shared_candidate_pool, FEATURE_PROVENANCE, RSQ2_TARGET)
six_rsq2_spearman = rank_by_target_correlation(six_rsq2_table, six_rsq2_dedup, RSQ2_TARGET)
print("6survey RSQ2 top-5 by Spearman (informational only):")
print(six_rsq2_spearman.head(5).to_string(index=False))
print("\n4survey RSQ2 top-5 by combined rank (the actual exported Set2):")
print(rsq2_combined.head(5).to_string(index=False))


6survey RSQ2 top-5 by Spearman (informational only):
              variable  spearman_rho  abs_rho
   elevation_roughness      0.275207 0.275207
     local_relief_500m      0.272532 0.272532
         slope_degrees      0.270226 0.270226
chelsa_bio12_precip_mm     -0.195075 0.195075
                 topex      0.132841 0.132841

4survey RSQ2 top-5 by combined rank (the actual exported Set2):
              variable  abs_rho  r2_drop_from_permutation  r2_drop_from_removal  spearman_rank  permutation_rank  ablation_rank  average_rank
chelsa_bio12_precip_mm 0.363904                  0.106964              0.060221            2.0               2.0           12.0      5.333333
      chelsa_gdd5_degc 0.310277                  0.032647              0.078154            5.0               5.0            8.0      6.000000
     gwa_weibull_k_50m 0.263665                  0.020188              0.152693            8.0               8.0            3.0      6.333333
         slope_degrees 0.086517       

## 11. Export -- copy-pasteable Set1-5 definitions, plus a manifest CSV

The dict literals below (as printed, not retyped by hand) are the copy-paste source for
`ENV_TERRAIN_FEATURE_SETS` (RSQ1), new `FEATURE_SETS` entries for `xgb_environmental.py` (RSQ2),
and a new `NESTED_FEATURE_SETS` dict for `broad_environmental_check.py` (RSQ3) -- existing entries
in all three files are never removed or edited, these are new, additive options. The manifest CSV
is a machine-written audit trail: hand-transcribing up to 15 lists across 3 destination files is
exactly where a missed/extra column is easy to make and hard to spot.


In [24]:
rsq2_export = {
    "nested_set1_baseline": SET1_BASELINE_COLUMNS,
    "nested_set2_top10": rsq2_set2,
    "nested_set3_gated_terrain_wind_vif": rsq2_set3_final,
    "nested_set4_gated_all_vif": rsq2_set4_final,
    "nested_set5_all_ungated_vif": rsq2_set5_final,
}
rsq3_export = {
    "nested_set1_baseline": SET1_BASELINE_COLUMNS,
    "nested_set2_top10": rsq3_set2,
    "nested_set3_gated_terrain_wind_vif": rsq3_set3_final,
    "nested_set4_gated_all_vif": rsq3_set4_final,
    "nested_set5_all_ungated_vif": rsq3_set5_final,
}

print("# --- RSQ1: new entries for torch_data.py's ENV_TERRAIN_FEATURE_SETS (baseline stripped) ---")
for name, columns in rsq1_export.items():
    print(f'"{name}": {columns!r},')

print("\n# --- RSQ2: new entries for xgb_environmental.py's FEATURE_SETS ---")
for name, columns in rsq2_export.items():
    print(f'"{name}": {columns!r},')

print("\n# --- RSQ3: new NESTED_FEATURE_SETS dict for broad_environmental_check.py ---")
for name, columns in rsq3_export.items():
    print(f'"{name}": {columns!r},')


# --- RSQ1: new entries for torch_data.py's ENV_TERRAIN_FEATURE_SETS (baseline stripped) ---
"nested_set2_top10": ['gwa_weibull_k_50m', 'dist_to_scpt_boundary', 'elevation', 'tas_mean', 'eastness', 'windward_topex', 'gwa_weibull_a_10m', 'gwa_weibull_a_50m', 'slope_degrees', 'cpmt_compactness_ratio'],
"nested_set3_gated_terrain_wind_vif": ['gwa_weibull_k_50m', 'elevation', 'eastness', 'windward_topex', 'gwa_weibull_a_10m', 'gwa_weibull_a_50m', 'slope_degrees', 'gwa_weibull_k_10m', 'solar_radiation_index', 'ceh_twi', 'gwa_wind_speed_10m'],
"nested_set4_gated_all_vif": ['gwa_weibull_k_50m', 'eastness', 'windward_topex', 'gwa_weibull_a_10m', 'gwa_weibull_a_50m', 'slope_degrees', 'gwa_weibull_k_10m', 'solar_radiation_index', 'ceh_twi', 'gwa_wind_speed_10m', 'dist_to_scpt_boundary', 'tas_mean', 'chelsa_gdd5_degc', 'cpmt_compactness_ratio'],
"nested_set5_all_ungated_vif": ['slope_degrees', 'northness', 'eastness', 'profile_curvature', 'plan_curvature', 'local_relief_500m', 'frost_hollow_flag'

In [25]:
manifest_rows = []
combined_lookup = {"RSQ1": rsq1_combined, "RSQ2": rsq2_combined, "RSQ3": rsq3_combined}
export_lookup = {
    "RSQ1": {"nested_set1_baseline": SET1_BASELINE_COLUMNS, **rsq1_export},
    "RSQ2": rsq2_export,
    "RSQ3": rsq3_export,
}
for rsq, sets in export_lookup.items():
    combined_indexed = combined_lookup[rsq].set_index("variable")
    for set_name, columns in sets.items():
        for column in columns:
            row = {
                "rsq": rsq,
                "set_name": set_name,
                "column": column,
                "is_baseline": column in SET1_BASELINE_COLUMNS,
            }
            if column in combined_indexed.index:
                row["spearman_abs_rho"] = combined_indexed.loc[column, "abs_rho"]
                row["permutation_r2_drop"] = combined_indexed.loc[column, "r2_drop_from_permutation"]
                row["ablation_r2_drop"] = combined_indexed.loc[column, "r2_drop_from_removal"]
                row["average_rank"] = combined_indexed.loc[column, "average_rank"]
            manifest_rows.append(row)
manifest_df = pd.DataFrame(manifest_rows)
manifest_path = project_root / "documentation" / "env_feature_sets_manifest.csv"
manifest_df.to_csv(manifest_path, index=False)
print(f"Saved {manifest_path} ({len(manifest_df)} rows)")


Saved /Users/shreeyagorasia/UoE_Docs/Dissertation/forest_diss/documentation/env_feature_sets_manifest.csv (225 rows)


## Findings

- **Set2 is now VIF-screened with backfill**, fixing a real gap found in review: previously
  Set2 (top-10 by importance rank) had no collinearity check at all, so a collinear pair could
  both survive if they independently ranked highly. `build_set2` now walks the ranked list in
  order, skips a candidate if its VIF against baseline + already-kept candidates exceeds 5.0, and
  tries the next-ranked candidate instead -- so Set2 still reaches 10 members (unless the ranked
  list runs out).
- **Confirmed fixed**: RSQ1's `dist_to_scpt_boundary`/`dist_to_cpmt_boundary` pair (mutually
  VIF~9.14, the original finding that started this) no longer both appear in Set2.
  `dist_to_scpt_boundary` was kept (ranked higher, added first); `dist_to_cpmt_boundary` was
  skipped (VIF=5.97 once `dist_to_scpt_boundary` was already in). Backfilled with `slope_degrees`
  and `cpmt_compactness_ratio`.
- **RSQ2 Set2**: `elevation` skipped (VIF=9.65), backfilled with `tas_mean`.
- **RSQ3 Set2**: `chelsa_gdd5_degc` (VIF=9.46), `chelsa_bio12_precip_mm` (VIF=5.40), and
  `tas_mean` (VIF=6.04) all skipped -- these three climate variables are mutually collinear with
  each other and with what was already kept; backfilled with `dist_to_road`, `topex`,
  `solar_radiation_index`, `soilgrids_ph`.
- **Final set sizes** (Set2 = 14 for all three RSQs -- baseline=4 + 10 VIF-clean candidates):

  | Set  | RSQ1 | RSQ2 | RSQ3 |
  |------|------|------|------|
  | Set1 | 4    | 4    | 4    |
  | Set2 | 14   | 14   | 14   |
  | Set3 | 15   | 14   | 12   |
  | Set4 | 18   | 19   | 20   |
  | Set5 | 30   | 28   | 31   |

**What this means for what's next.** Set2/3/4/5 are all now VIF-screened for all three RSQs --
the collinearity concern raised earlier in review is resolved everywhere, not just Set3/4/5.
Still not committed to `xgb_environmental.py`'s `FEATURE_SETS` or `broad_environmental_check.py`
(RSQ2/RSQ3) -- only RSQ1's `ENV_TERRAIN_FEATURE_SETS` in `torch_data.py` reads live from the
manifest so far. The manifest CSV (`documentation/env_feature_sets_manifest.csv`) is the current
source of truth for all 15 sets.
